# 3. What result would answer the question?

**Goal:** distinguish absolute accuracy from degradation, then understand the unit resampled in the uncertainty calculation.

All scores and identities are invented. This is a small statistical teaching example, not a scientific result or production report generator. It uses NumPy only and never reads, writes, or downloads study data.

## Define the effect before seeing results

For method $m$, seed $s$, and corrupted condition $c$,
\[
d_{m,s,c}=A^{\rm clean}_{m,s}-A^{\rm corrupt}_{m,s,c},\qquad
\Delta=\frac{1}{3\cdot9}\sum_{s,c}(d_{\rm contextual,s,c}-d_{\rm contrastive,s,c}).
\]
Accuracy $A$ is the fraction of queries with the correct top-ranked exemplar. Each query has equal weight within its cell; seeds and the nine conditions have equal weight. Negative $\Delta$ means smaller degradation for contextual training.

A method can lose fewer percentage points simply because it starts lower. This counterexample uses one aggregate score per method to make the distinction visible.

In [1]:
import numpy as np

clean = np.array([0.70, 0.90])       # contextual, contrastive
corrupted = np.array([0.65, 0.75])
drops = clean - corrupted
delta = drops[0] - drops[1]
clean_gap = clean[0] - clean[1]
corrupted_gap = corrupted[0] - corrupted[1]
assert np.isclose(delta, clean_gap - corrupted_gap)
assert delta < 0 and corrupted_gap < 0
print('Clean accuracies (%):', 100 * clean)
print('Corrupted accuracies (%):', 100 * corrupted)
print('Drops (percentage points):', np.round(100 * drops, 2))
print(f'Delta: {100 * delta:.1f} percentage points')
print(f'Corrupted-accuracy gap: {100 * corrupted_gap:.1f} percentage points')
print('Contextual degrades less but still retrieves fewer corrupted queries correctly.')

Clean accuracies (%): [70. 90.]
Corrupted accuracies (%): [65. 75.]
Drops (percentage points): [ 5. 15.]
Delta: -10.0 percentage points
Corrupted-accuracy gap: -10.0 percentage points
Contextual degrades less but still retrieves fewer corrupted queries correctly.


## Build the paired per-query effect

Correctness is 0 or 1. The same query is scored under both methods, all three seeds, and every condition. For query $i$, average its paired difference of drops:
\[
e_i=\frac{1}{27}\sum_{s,c}
[(y^{\rm clean}_{C,s,i}-y^{\rm corrupt}_{C,s,c,i})
-(y^{\rm clean}_{B,s,i}-y^{\rm corrupt}_{B,s,c,i})].
\]
Here $C$ denotes contextual and $B$ standard contrastive. Then $\Delta=N^{-1}\sum_i e_i$. Linearity of averaging makes this equal to averaging the cell-level accuracy-drop differences.

Keep three seeds and nine conditions, but shrink the data to **two actions, three performances per action, and twelve queries**. Camera-view counts differ. This size teaches the arithmetic; it cannot establish reliable scientific uncertainty. The invented correctness generator is not a model of actual camera errors or retrieval.

In [2]:
seeds = (7, 17, 29)
# Performance key = (setup, performer, repetition, action); camera is excluded.
group_keys = [(1, 101, 1, 11), (1, 101, 2, 11), (1, 102, 1, 11),
              (2, 201, 1, 22), (2, 202, 1, 22), (2, 202, 2, 22)]
group_sizes = np.array([3, 1, 2, 1, 3, 2])
query_group = np.repeat(np.arange(len(group_keys)), group_sizes)
rng = np.random.Generator(np.random.PCG64(42))
# Shared draws simplify this invented comparison; they are not real corruption draws.
uniforms = rng.random((3, 10, len(query_group)))
probabilities = np.array([
    [0.76, 0.72, 0.68, 0.64, 0.72, 0.68, 0.64, 0.72, 0.68, 0.64],
    [0.90, 0.81, 0.72, 0.63, 0.81, 0.72, 0.63, 0.81, 0.72, 0.63],
])
# Axis order: method, seed, condition (clean then nine corruptions), query.
correct = (uniforms[None, :, :, :] < probabilities[:, None, :, None]).astype(float)
effects = ((correct[0, :, 0, :][:, None, :] - correct[0, :, 1:, :])
           - (correct[1, :, 0, :][:, None, :] - correct[1, :, 1:, :]))
per_query = effects.mean(axis=(0, 1))
cell_effects = effects.mean(axis=2)
estimate = per_query.mean()
assert correct.shape == (2, 3, 10, 12)
assert effects.shape == (3, 9, 12)
assert np.isclose(estimate, cell_effects.mean())
print('Correctness array shape:', correct.shape)
print(f'Toy Delta: {100 * estimate:.2f} percentage points')
print('Per-seed effects:', np.round(100 * effects.mean(axis=(1, 2)), 2))

Correctness array shape: (2, 3, 10, 12)
Toy Delta: -17.28 percentage points
Per-seed effects: [-26.85   2.78 -27.78]


## Resample performances; weight queries

Synchronized camera views are related observations of one performance. Resampling each view independently breaks that dependence. Draw performances with replacement and retain **all their views**, with multiplicity.

This does not mean equally averaging performance means. If one performance has three queries and another has one, the query-weighted target gives the first three times as much weight. For sampled groups $g_1,\ldots,g_K$,
\[
\Delta^*=\frac{\sum_j\sum_{i\in g_j}e_i}{\sum_j|g_j|}.
\]
The denominator can change when differently sized groups are sampled.

In [3]:
demonstration_effects = [np.array([-0.2, -0.2, -0.2]), np.array([0.2])]
query_weighted = np.concatenate(demonstration_effects).mean()
equal_group_weighted = np.mean([values.mean() for values in demonstration_effects])
assert np.isclose(query_weighted, -0.1)
assert np.isclose(equal_group_weighted, 0.0)
print(f'Query-weighted effect: {100 * query_weighted:.1f} percentage points')
print(f'Equal average of group means: {100 * equal_group_weighted:.1f} percentage points')
print('These are different targets; the study uses the first.')

Query-weighted effect: -10.0 percentage points
Equal average of group means: -0.0 percentage points
These are different targets; the study uses the first.


## Bootstrap within each fixed action

A bootstrap approximates repeated sampling by drawing observed groups with replacement. Action identities stay fixed. Within **each** action, draw its original number of performance groups. Do not resample seeds. Apply the same query multiplicities to every method, seed, and condition.

Because `per_query` already averages the paired matrix, one shared draw preserves all these pairings. The code also checks one draw against indexing the unaveraged matrix directly. The project uses 10,000 draws from PCG64(2026) and a 95% percentile interval with linear quantiles; this toy uses the same settings.

In [4]:
group_rows = [np.flatnonzero(query_group == g) for g in range(len(group_keys))]
group_sums = np.array([per_query[rows].sum() for rows in group_rows])
actions = sorted({key[3] for key in group_keys})
groups_by_action = [np.array([g for g, key in enumerate(group_keys) if key[3] == action])
                    for action in actions]
bootstrap_rng = np.random.Generator(np.random.PCG64(2026))
replicates = np.empty(10_000)
for replicate in range(len(replicates)):
    sampled_groups = np.concatenate([
        groups[bootstrap_rng.integers(0, len(groups), size=len(groups))]
        for groups in groups_by_action
    ])
    replicates[replicate] = group_sums[sampled_groups].sum() / group_sizes[sampled_groups].sum()
    if replicate == 0:
        sampled_queries = np.concatenate([group_rows[g] for g in sampled_groups])
        assert np.isclose(replicates[replicate], effects[:, :, sampled_queries].mean())
        for action, groups in zip(actions, groups_by_action):
            assert sum(group_keys[g][3] == action for g in sampled_groups) == len(groups)
lower, upper = np.quantile(replicates, [0.025, 0.975], method='linear')
assert np.isfinite(replicates).all()
assert lower <= upper
print(f'Toy estimate: {100 * estimate:.2f} percentage points')
print(f'Toy conditional 95% interval: [{100 * lower:.2f}, {100 * upper:.2f}] percentage points')
print('Seeds, actions, gallery, and realized corruption outcomes stayed fixed.')

Toy estimate: -17.28 percentage points
Toy conditional 95% interval: [-29.88, -5.25] percentage points
Seeds, actions, gallery, and realized corruption outcomes stayed fixed.


## Match the claim to the uncertainty

The study supports its one confirmatory claim only when the interval's **upper bound is below zero**. That supports smaller degradation under the declared mixture, not superiority in every condition. An interval including zero does not establish equivalence. An entirely positive interval favors greater contextual degradation.

The interval assumes performances are exchangeable within each fixed action. It is conditional on trained heads, seeds, actions, anchors, query definitions, and realized artificial corruptions. Repeated performances by one person may remain dependent. It does not quantify uncertainty across arbitrary people, new seeds, actions, anchors, or real camera errors. Ten thousand resamples cannot create those missing sources of variation.

In [5]:
if upper < 0:
    interpretation = 'Smaller degradation in this synthetic example; inspect absolute accuracy too.'
elif lower > 0:
    interpretation = 'Greater contextual degradation in this synthetic example.'
else:
    interpretation = 'Inconclusive for smaller degradation; not evidence of equivalence.'
print(interpretation)
print('This notebook makes no claim about the real experiment.')

Smaller degradation in this synthetic example; inspect absolute accuracy too.
This notebook makes no claim about the real experiment.


## Self-checks and answers

1. **Why average paired effects before resampling?** It equals applying the same group draw to the fixed full matrix and makes the target explicit.
2. **Why not resample seeds?** The frozen analysis conditions on those seeds. It does not claim population-level uncertainty over training randomness.
3. **Why not give actions equal weight?** Queries have equal weight. Drawing each action's original number of groups fixes the strata without changing the target to an average of action means.
4. **What if a performance is sampled twice?** Include all its views twice in numerator and denominator.
5. **What belongs beside the interval?** Clean and absolute corrupted accuracy, all seed and condition effects, and the fixed leave-one-action-out sensitivity. That sensitivity removes an action's queries while keeping the full gallery and original ranks.
6. **Does an interval validate the pipeline?** No. Inference still depends on locked data, correct ranks, paired training, and independent numerical/provenance checks.

The governing source is [the final v3 plan, Section 6: Locked analysis](../../plan/research-plan.v3.md#6-locked-analysis). Production analysis is in [evaluation/analysis.py](../../src/pose_embed/evaluation/analysis.py).